# Notebook 5: Level 3 - Full Real-Time RAG on OpenShift AI

**Everything combined:**
- **Batch features:** Customer profile (name, plan, coverage)
- **Streaming features:** Live claim status via PushSource
- **On-demand features:** Computed at request time (`estimated_payout`, `can_withdraw_claim`, `days_since_filed`)
- **Conversation memory:** Read/write interaction history
- **OpenAI-compatible search API:** `store.openai_search()` for auto-embedding
- **Hybrid search:** Vector + full-text combined
- **MCP Server:** Feast exposes all capabilities as MCP tools for Dify.ai

This is the full-featured RAG pipeline that Feast + OpenShift AI enables.

In [ ]:
import os, sys
REPO_DIR = os.path.expanduser('~/life-insurance-rag')
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'app'))

from feast import FeatureStore
from sentence_transformers import SentenceTransformer
import pandas as pd
from datetime import datetime

store = FeatureStore(repo_path='feature_repo')
model = SentenceTransformer('all-MiniLM-L6-v2')

## Step 1: On-Demand Feature Views

On-demand features are computed at request time by combining inputs from other feature views.

```python
@on_demand_feature_view(
    sources=[customer_profile, claim_status],
    schema=[...],
)
def realtime_computed(inputs):
    df['estimated_payout'] = inputs['coverage_amount']
    df['can_withdraw_claim'] = 'yes' if early stage else 'no'
    df['days_since_filed'] = (now - filed_date).days
    return df
```

In [ ]:
customer_id = 'POL-1001'

# Fetch ALL features including on-demand computed ones
all_features = store.get_online_features(
    features=[
        # Batch
        'customer_profile:customer_name',
        'customer_profile:plan_type',
        'customer_profile:coverage_amount',
        # Streaming
        'claim_status:claim_stage',
        'claim_status:claim_filed_date',
        'claim_status:documents_pending',
        # On-demand (computed NOW from the above)
        'realtime_computed:estimated_payout',
        'realtime_computed:can_withdraw_claim',
        'realtime_computed:days_since_filed',
    ],
    entity_rows=[{'customer_id': customer_id}],
).to_dict()

print('--- All Features (Batch + Streaming + On-Demand) ---')
for k, v in all_features.items():
    if k != 'customer_id':
        print(f'  {k}: {v[0]}')

## Step 2: Conversation Memory (Read)

In [ ]:
memory = store.get_online_features(
    features=[
        'interaction_memory:last_topic',
        'interaction_memory:last_resolution',
        'interaction_memory:visit_count',
        'interaction_memory:open_question',
    ],
    entity_rows=[{'customer_id': customer_id}],
).to_dict()

print('--- Conversation Memory ---')
for k, v in memory.items():
    if k != 'customer_id':
        print(f'  {k}: {v[0]}')

## Step 3: OpenAI-Compatible Search API

`store.openai_search()` accepts plain text, auto-embeds using the configured
`embedding_model` in `feature_store.yaml`, and returns OpenAI-compatible results.

In [ ]:
try:
    openai_results = store.openai_search(
        query='What is my claim status?',
        top_k=3,
        feature='insurance_docs:embedding',
    )
    print('=== OpenAI-Compatible Search Results ===')
    for r in openai_results:
        print(f'  Score: {r.get("score", "N/A")}')
        print(f'  Text: {str(r.get("text", ""))[:120]}...')
        print()
except Exception as e:
    print(f'OpenAI search: {e}')
    print('(Requires embedding_model config in feature_store.yaml)')

## Step 4: Full Level 3 Prompt

In [ ]:
query = 'Can I withdraw my claim? What will the payout be?'
query_vector = model.encode(query).tolist()

# Hybrid search
results = store.retrieve_online_documents(
    feature='insurance_docs:embedding',
    query=query_vector,
    top_k=3,
)
docs_df = results.to_df()

doc_context = '\n---\n'.join(
    f'{row["title"]}: {row["content"]}'
    for _, row in docs_df.iterrows()
)

name = all_features['customer_name'][0]
visit_count = memory.get('visit_count', [0])[0] or 0
last_topic = memory.get('last_topic', [None])[0]

memory_section = f'Visit count: {visit_count}'
if last_topic:
    memory_section += f', Last topic: {last_topic}'

prompt = f"""You are a life insurance assistant with full context.

MEMORY: {memory_section}

CUSTOMER: {name}
  Plan: {all_features['plan_type'][0]}
  Coverage: Rs {all_features['coverage_amount'][0]:,.0f}
  Claim stage: {all_features['claim_stage'][0]}
  Est. payout: Rs {all_features['estimated_payout'][0]:,.0f}
  Can withdraw: {all_features['can_withdraw_claim'][0]}
  Days since filed: {all_features['days_since_filed'][0]}

DOCUMENTS:
{doc_context}

QUESTION: {query}

Acknowledge their previous interaction and answer with full context."""

print('=== Level 3: Full Context Prompt ===')
print(f'Features used: batch + streaming + on-demand + memory')
print(f'Search type: hybrid (vector + full-text)')
print(f'On-demand computed: estimated_payout={all_features["estimated_payout"][0]}, can_withdraw={all_features["can_withdraw_claim"][0]}')
print()
print(prompt[:800])
print('...')

## Step 5: Write Memory Back (for Next Interaction)

In [ ]:
memory_update = pd.DataFrame({
    'customer_id': [customer_id],
    'last_topic': ['claim_withdrawal'],
    'last_resolution': ['informed_about_withdrawal_and_payout'],
    'visit_count': [visit_count + 1],
    'open_question': ['none'],
    'event_timestamp': [datetime.now()],
})

store.write_to_online_store('interaction_memory', memory_update)
print('Memory updated for next interaction.')

# Verify
updated = store.get_online_features(
    features=['interaction_memory:last_topic', 'interaction_memory:visit_count'],
    entity_rows=[{'customer_id': customer_id}],
).to_dict()
print(f'  last_topic: {updated["last_topic"][0]}')
print(f'  visit_count: {updated["visit_count"][0]}')

## Step 6: Verify the MCP Server (Feast Operator-managed)

The Feast operator deploys an MCP server alongside the online store.
Dify.ai connects to this via the MCP SSE endpoint.

In [ ]:
import os, json

feast_url = os.environ.get(
    'FEAST_SERVER_URL',
    'http://feast-life-insurance-rag-online.life-insurance-rag.svc.cluster.local:80'
)

print('=== Feast MCP Server (Operator-Managed) ===')
print(f'Online server: {feast_url}')
print(f'MCP SSE endpoint: {feast_url}/mcp/sse')
print()
print('To connect from Dify.ai:')
print('  1. Add an MCP Tool node in your Dify workflow')
print(f'  2. Set SSE URL: {feast_url}/mcp/sse')
print('  3. Available tools: get_online_features, retrieve_online_documents, push, etc.')
print()

# Test health
try:
    import urllib.request
    resp = urllib.request.urlopen(f'{feast_url}/health', timeout=5)
    print(f'Health check: {resp.status} OK')
except Exception as e:
    print(f'Health check: {e}')

## Summary: The RAG Maturity Journey

| Level | Feast Capabilities | Answer Quality |
|-------|-------------------|----------------|
| **0** | `retrieve_online_documents` (pgvector) | Generic, same for everyone |
| **1** | + `get_online_features` (batch) + metadata filter | Personalized by plan type |
| **2** | + `push()` (streaming) + hybrid search | Real-time claim awareness |
| **3** | + `@on_demand_feature_view` + `write_to_online_store` + `openai_search()` | Fully contextual |

### OpenShift AI Integration

- **Feast Operator** (managed by RHOAI DataScienceCluster) deploys and manages the feature store
- **FeatureStore CR** defines the entire deployment declaratively
- **MCP Server** auto-deployed by the operator for Dify.ai integration
- **PostgreSQL + pgvector** as the unified online store
- **RHOAI Workbench** for running notebooks and experiments
- **Shakti Cloud** provides sovereign GPU infrastructure for LLM inference

---

## Architecture on OpenShift AI

```
User Query
    |
    v
Dify.ai Workflow  <-- MCP SSE --> Feast Online Server (operator-managed)
    |                                    |
    |                         +----------+----------+
    |                         |          |          |
    |                    Batch FV   Stream FV   OnDemand FV
    |                    (profile)  (claims)   (computed)
    |                         |          |          |
    |                         +----+-----+----------+
    |                              |
    |                   PostgreSQL + pgvector
    |                    (online store)
    |                              |
    |            +-----------------+-----------------+
    |            |                 |                 |
    |       Vector Search    Full-Text          Hybrid
    |       (embedding)      (tsvector)         (both)
    |            |                 |                 |
    |            +--------+--------+-----------------+
    |                     |
    v                     v
LLM (Shakti Cloud / vLLM on RHOAI)  <-- context
    |
    v
Personalized Answer + Memory Write-back


OpenShift AI Cluster
+-----------------------------------------------------------+
| Namespace: life-insurance-rag                             |
|                                                           |
| [FeatureStore CR] --> Feast Operator                      |
|    |                                                      |
|    +-> feast-*-registry (gRPC + REST + MCP)               |
|    +-> feast-*-online   (REST + MCP)                      |
|                                                           |
| [PostgreSQL + pgvector]  (Deployment + PVC)               |
| [RHOAI Workbench]       (Notebook CR)                     |
+-----------------------------------------------------------+
```